# HAM10000 from Kaggle — CNN and Monte Carlo Dropout

Open this notebook in **Google Colab**, select **Runtime → Change runtime type → GPU**, and run the cells in order. The first cell installs KaggleHub and downloads [Skin Cancer MNIST: HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000). Allow several GB of disk space and time for the initial download. No Google Drive mount, manually uploaded images, or local metadata paths are required. Colab provides the TensorFlow, NumPy, pandas, Pillow, SciPy, scikit-learn, Matplotlib, and Seaborn dependencies used here.

The public download is attempted without credentials. If Kaggle asks for authentication, use `import kagglehub; kagglehub.login()` in a separate cell, or configure a Colab secret named `KAGGLE_API_TOKEN`, then rerun the download cell. Accept any dataset consent request on Kaggle if prompted. Do not paste credentials into saved notebook cells. See [KaggleHub authentication documentation](https://github.com/Kaggle/kagglehub#authenticate).

The original 224×224 RGB CNN, seven classes, 500 resampled images per class, normalization, 75/25 split, and 50-pass MC Dropout evaluation are retained. Images from both archive parts are discovered recursively and checked against metadata. Training now explicitly uses the original intended 10 epochs; batches of 16 and float32 arrays reduce memory use.

Unfinished PyTorch/CIFAR-10 examples and repeated model-reset/evaluation cells were removed so Run all evaluates the trained HAM10000 model once. Entropy, predictive variance, accuracy, and expected calibration error are preserved.

**Evaluation limitation inherited from the source:** balancing with replacement occurs before the random image-level split, so duplicate images and images of the same lesion can cross train/test boundaries. Results from this preserved experimental split should not be treated as an independent performance estimate. A research evaluation should split by lesion before resampling the training set.

In [ ]:
%pip -q install kagglehub

from pathlib import Path
import kagglehub

DATASET_HANDLE = "kmader/skin-cancer-mnist-ham10000"
# Public download; reuses KaggleHub's cache on subsequent runs.
# If Kaggle requests authentication/consent, see the instructions above.
DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
print("Kaggle dataset directory:", DATASET_ROOT)

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from glob import glob
from PIL import Image

In [ ]:
metadata_files = sorted(DATASET_ROOT.rglob("HAM10000_metadata.csv"))
if not metadata_files:
    raise FileNotFoundError(f"HAM10000_metadata.csv not found under {DATASET_ROOT}")
METADATA_PATH = metadata_files[0]
metadata_df = pd.read_csv(METADATA_PATH)
required_columns = {"image_id", "dx", "age", "sex", "localization"}
missing_columns = required_columns - set(metadata_df.columns)
if missing_columns:
    raise ValueError(f"Missing metadata columns: {sorted(missing_columns)}")
if metadata_df["image_id"].isna().any() or metadata_df["image_id"].duplicated().any():
    raise ValueError("Metadata must contain one non-null row per image_id.")
skin_df = metadata_df.copy()
print(f"Metadata: {METADATA_PATH}; {len(skin_df):,} images")

In [ ]:
# Search recursively: handles both image parts and nested Kaggle directories.
# Some distributions repeat the same image IDs in nested folders; keep one path.
image_path = {}
for path in sorted(DATASET_ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        image_path.setdefault(path.stem, str(path))
print(f"Indexed {len(image_path):,} unique image IDs")

In [ ]:
skin_df["path"] = skin_df["image_id"].map(image_path)
missing_ids = skin_df.loc[skin_df["path"].isna(), "image_id"].tolist()
if missing_ids:
    raise FileNotFoundError(
        f"{len(missing_ids)} metadata images are missing (examples: {missing_ids[:5]}). "
        "Check the Kaggle download; do not silently discard an image part."
    )
metadata_df = skin_df.copy()

In [ ]:
IMAGE_SIZE = (224, 224)

def load_rgb(path):
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB").resize(IMAGE_SIZE), dtype=np.uint8)

# Load only plotted images here; avoid storing 10,015 full-size arrays for EDA.

In [ ]:
print(skin_df['dx'].value_counts())
#print(skin_df.head(10))
print(skin_df.tail(10))

In [ ]:
import matplotlib.pyplot as plt
#import numpy as np
n_samples = 5

fig, m_axs = plt.subplots(7, n_samples, figsize = (4*n_samples, 3*7))
for n_axs, (type_name, type_rows) in zip(m_axs, skin_df.sort_values(['dx']).groupby('dx')):

    n_axs[0].set_title(type_name)
    for c_ax, (_, c_row) in zip(n_axs, type_rows.sample(n_samples, random_state=1234).iterrows()):
        c_ax.imshow(load_rgb(c_row['path']))
        c_ax.axis('off')

In [ ]:
np.random.seed(42)
from sklearn.metrics import confusion_matrix
#import itertools

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from sklearn.model_selection import train_test_split
from scipy import stats
from sklearn.preprocessing import LabelEncoder

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
skin_df = metadata_df.copy()  # Reuse validated Kaggle metadata and paths.

In [ ]:
SIZE = IMAGE_SIZE[0]  # The existing CNN expects 224 x 224 RGB images.

In [ ]:
#le = LabelEncoder()
#le.fit(skin_df['dx'])
#labelEncoder()
#print(list(le.classes_))
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(skin_df['dx'])
print(list(le.classes_))


In [ ]:
skin_df['label'] = le.transform(skin_df["dx"])
print(skin_df.sample(10))

In [ ]:
fig = plt.figure(figsize=(15, 10))
#cols = 5
#rows = 3

In [ ]:
ax1 = fig.add_subplot(221)
skin_df['dx'].value_counts().plot(kind='bar', ax=ax1)
ax1.set_ylabel('Count')
ax1.set_title('Cell Type')

In [ ]:
ax2=fig.add_subplot(222)
skin_df['sex'].value_counts().plot(kind='bar', ax=ax2)
ax2.set_ylabel('Count', size=15)
ax2.set_title('sex')

In [ ]:
ax3 = fig.add_subplot(223)
skin_df['localization'].value_counts().plot(kind='bar')
ax3.set_ylabel('Count', size=12)
ax3.set_title('localization')

In [ ]:
#ax4 = fig.add_subplot(224)
#sample_age = skin_df[pd.notnull(skin_df['age'])]
#sns.distplot(sample_age['age'],fit=stats.norm, color='red')
#ax4.set_title('age')
#plt.tight_layout()
#plt.show()
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

fig = plt.figure(figsize=(10,6))
ax4 = fig.add_subplot(224)

# Drop NaN values
sample_age = skin_df[pd.notnull(skin_df['age'])]

# Plot histogram with KDE
sns.histplot(sample_age['age'], kde=True, stat="density", color='red', ax=ax4)

# Fit a normal distribution
mu, sigma = stats.norm.fit(sample_age['age'])
xmin, xmax = ax4.get_xlim()
x = np.linspace(xmin, xmax, 100)
p = stats.norm.pdf(x, mu, sigma)

# Plot the fitted curve
ax4.plot(x, p, 'k', linewidth=2)

ax4.set_title('Age distribution')
plt.show()


In [ ]:
#distribution of data into various classes
from sklearn.utils import resample
print(skin_df['label'].value_counts())

In [ ]:
df_0 = skin_df[skin_df['label'] == 0]
df_1 = skin_df[skin_df['label'] == 1]
df_2 = skin_df[skin_df['label'] == 2]
df_3 = skin_df[skin_df['label'] == 3]
df_4 = skin_df[skin_df['label'] == 4]
df_5 = skin_df[skin_df['label'] == 5]
df_6 = skin_df[skin_df['label'] == 6]

In [ ]:
n_samples = 500
df_0_balanced = resample(df_0, replace=True, n_samples=n_samples, random_state=42)
df_1_balanced = resample(df_1, replace=True, n_samples=n_samples, random_state=42)
df_2_balanced = resample(df_2, replace=True, n_samples=n_samples, random_state=42)
df_3_balanced = resample(df_3, replace=True, n_samples=n_samples, random_state=42)
df_4_balanced = resample(df_4, replace=True, n_samples=n_samples, random_state=42)
df_5_balanced = resample(df_5, replace=True, n_samples=n_samples, random_state=42)
df_6_balanced = resample(df_6, replace=True, n_samples=n_samples, random_state=42)



In [ ]:
skin_df_balanced = pd.concat([df_0_balanced, df_1_balanced,
                     df_2_balanced, df_3_balanced,
                     df_4_balanced, df_5_balanced, df_6_balanced])


In [ ]:
print(skin_df_balanced['label'].value_counts())


In [ ]:
skin_df_balanced = skin_df_balanced.copy()
skin_df_balanced['path'] = skin_df_balanced['image_id'].map(image_path)
if skin_df_balanced['path'].isna().any():
    raise FileNotFoundError("Balanced data contains unresolved image paths.")

In [ ]:
# Decode each unique balanced image once (resampling can repeat IDs).
image_cache = {
    path: load_rgb(path) for path in skin_df_balanced['path'].unique()
}
skin_df_balanced['image'] = skin_df_balanced['path'].map(image_cache)
del image_cache

n_samples = 5
fig, m_axs = plt.subplots(7, n_samples, figsize=(4 * n_samples, 3 * 7))
for n_axs, (type_name, type_rows) in zip(m_axs, skin_df_balanced.sort_values(['dx']).groupby('dx')):
    n_axs[0].set_title(type_name)
    for c_ax, (_, c_row) in zip(n_axs, type_rows.sample(n_samples, random_state=1234).iterrows()):
        c_ax.imshow(c_row['image'])
        c_ax.axis('off')
plt.show()

In [ ]:
# Keep the original balancing and 75/25 split, with float32 normalization.
expected_shape = (*IMAGE_SIZE, 3)
if not skin_df_balanced['image'].map(lambda image: image.shape == expected_shape).all():
    raise ValueError(f"Every decoded image must have shape {expected_shape}")
valid_df = skin_df_balanced
X = np.stack(valid_df['image'].to_numpy()).astype(np.float32)
X /= np.float32(255.0)
Y = valid_df['label'].to_numpy()
Y_cat = to_categorical(Y, num_classes=7)
x_train, x_test, y_train, y_test = train_test_split(
    X, Y_cat, test_size=0.25, random_state=42
)
del X  # The split creates separate arrays; release the large original buffer.
skin_df_balanced.drop(columns='image', inplace=True)
print("Training:", x_train.shape, "Test:", x_test.shape, "dtype:", x_train.dtype)

In [ ]:
num_classes=7
model = Sequential() # Instantiate the Sequential model
model.add(Input(shape=(*IMAGE_SIZE, 3)))
model.add(Conv2D(256,(3,3),activation='relu'))

model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.3))

model.add(Conv2D(128,(3,3),activation='relu'))

model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.3))


model.add(Conv2D(64,(3,3),activation='relu'))

model.add(MaxPool2D(pool_size=(2,2)))
model.add(Dropout(0.3))
model.add(Flatten())

model.add(Dense(32))
model.add(Dense(num_classes, activation='softmax')) # Use num_classes variable

model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])

In [ ]:
batch_size = 16  # Reduce to 8 if your Colab GPU runs out of memory.
epochs = 10
history = model.fit(
    x_train, y_train, batch_size=batch_size, epochs=epochs,
    validation_data=(x_test, y_test),
    verbose=2)

score = model.evaluate(x_test, y_test, batch_size=batch_size)
print('Test accuracy:', score[1])

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'y', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()



acc = history.history['acc']
val_acc = history.history['val_acc']
plt.plot(epochs, acc, 'y', label='Training acc')
plt.plot(epochs, val_acc, 'r', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
y_pred = model.predict(x_test, batch_size=batch_size)

y_pred_classes = np.argmax(y_pred, axis=1)

y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred_classes)
fig, ax = plt.subplots(figsize=(6, 6))
sns.set(font_scale=1.4)
sns.heatmap(cm, annot=True, linewidths=.5, ax=ax)


incorr_fraction = 1 - np.diag(cm) / np.sum(cm, axis=1)
plt.bar(np.arange(7), incorr_fraction)
plt.xlabel('True Label')
plt.ylabel('Fraction of incorrect predictions')

## Monte Carlo Dropout on the trained model

Perform 50 stochastic passes in small batches, with dropout active (`training=True`). The model weights from training above are reused.

In [ ]:
import tensorflow as tf
from sklearn.metrics import accuracy_score
from scipy.stats import entropy

# ---------- Monte Carlo Dropout ----------
def mc_dropout_predictions(model, x, n_samples=50, batch_size=16):
    """
    Run Monte Carlo Dropout sampling
    Args:
        model: trained Keras model with Dropout layers
        x: input images (batch, 224, 224, 3)
        n_samples: number of stochastic forward passes
    Returns:
        mean_preds: average predictions (probabilities)
        var_preds: variance of predictions
        all_preds: all raw predictions from MC sampling
    """
    preds = []
    for _ in range(n_samples):
        # Force dropout to be active (training=True)
        # Bound activation memory; retain stochastic dropout in every batch.
        batch_preds = [
            model(x[start:start + batch_size], training=True).numpy()
            for start in range(0, len(x), batch_size)
        ]
        preds.append(np.concatenate(batch_preds, axis=0))
    preds = np.array(preds)  # shape: (n_samples, batch, n_classes)
    mean_preds = preds.mean(axis=0)
    var_preds = preds.var(axis=0)
    return mean_preds, var_preds, preds


# ---------- Expected Calibration Error (ECE) ----------
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Compute ECE = sum_k |acc(bin) - conf(bin)| * (bin_size / total)
    """
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = np.argmax(y_prob, axis=1)
    confidences = np.max(y_prob, axis=1)

    ece = 0.0
    bin_boundaries = np.linspace(0, 1, n_bins+1)

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i+1]
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            acc_in_bin = np.mean(y_true[in_bin] == y_pred[in_bin])
            avg_conf_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(acc_in_bin - avg_conf_in_bin) * prop_in_bin

    return ece


# ---------- Run MC Dropout on test set ----------
n_samples = 50
mean_preds, var_preds, all_preds = mc_dropout_predictions(model, x_test, n_samples=n_samples)

# Convert one-hot y_test to integers
y_true = np.argmax(y_test, axis=1)

# Predicted classes using MC mean
y_pred_mc = np.argmax(mean_preds, axis=1)

# Accuracy
acc_mc = accuracy_score(y_true, y_pred_mc)
print("Monte Carlo Dropout Accuracy:", acc_mc)

# ---------- Uncertainty Measures ----------
# Variance (per sample, across classes)
avg_variance = np.mean(var_preds)
print("Average predictive variance:", avg_variance)

# Entropy (per sample, then average)
entropies = entropy(mean_preds.T)  # entropy of each prediction
avg_entropy = np.mean(entropies)
print("Average predictive entropy:", avg_entropy)

# Expected Calibration Error (ECE)
ece = expected_calibration_error(y_true, mean_preds, n_bins=15)
print("Expected Calibration Error (ECE):", ece)